In [ ]:
#EL OBJETIVO DE ESTE EXPERIMENTO ES PREDECIR ETIQUETAS CON Y SIN AUMENTACIÓN DE DATOS EN EL TRAIN


import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
import ast
from collections import Counter


#CARGA DE DATOS
train_data = pd.read_csv("../../Data/OnlyOneEmotion/train_emotions.csv")
test_conflictive = pd.read_csv("../../Data/OnlyOneEmotion/test_emotions_conflicting.csv")
test_clean_data = pd.read_csv("../../Data/OnlyOneEmotion/test_emotions_complete.csv")

# Convertir las cadenas de listas en listas reales
all_emotions = test_conflictive['Emotion'].apply(ast.literal_eval)

# Aplanar todas las listas en una sola lista
flattened_emotions = [emotion for sublist in all_emotions for emotion in sublist]

# Contar ocurrencias de cada emoción
emotion_counts = Counter(flattened_emotions)

# Imprimir resultados
print("Distribución total de emociones en el test conflictivo (contando todas):\n")
for emotion, count in emotion_counts.items():
    print(f"{emotion}: {count}")


#Hacer un split de train_data para obtener un conjunto de validación
train, valid = train_test_split(train_data, test_size=0.2, random_state=42)

#Vectorizar los textos
vectorizer = TfidfVectorizer(max_features=10000, lowercase=True, strip_accents='unicode')

# Ajustar el vectorizador y transformar los conjuntos de datos
X_train = vectorizer.fit_transform(train['Text'])
X_valid = vectorizer.transform(valid['Text'])
X_test_conflictive = vectorizer.transform(test_conflictive['Text'])

y_train = train['Emotion']
y_valid = valid['Emotion']
y_test_conflictive = test_conflictive['Emotion']

#Modelos de clasificación
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'SVM (linear)': SVC(kernel='linear', class_weight='balanced'),
    'Decision Tree': DecisionTreeClassifier(random_state=42)
}

# Entrenamiento y evaluación de los modelos
for model_name, clf in models.items():
    print(f"\nEntrenando modelo: {model_name}")
    clf.fit(X_train, y_train)

    # Validación
    y_pred_valid = clf.predict(X_valid)
    print(f"\nPrecisión en validación ({model_name}): {accuracy_score(y_valid, y_pred_valid):.4f}")
    dict_valid = classification_report(y_valid, y_pred_valid, output_dict=True, zero_division=0)
    f1_macro_valid = dict_valid["macro avg"]["f1-score"]
    recall_macro_valid = dict_valid["macro avg"]["recall"]
    print(f"F1 Score en validación (macro avg): {f1_macro_valid:.4f}")
    print(f"Recall Score en validación (macro avg): {recall_macro_valid:.4f}")


    #Predicción en el conjunto de entrenamiento 
    y_pred_train = clf.predict(X_train)
    print(f"Precisión en entrenamiento ({model_name}): {accuracy_score(y_train, y_pred_train):.4f}")
    dict_train = classification_report(y_train, y_pred_train, output_dict=True, zero_division=0)
    f1_macro_train = dict_train["macro avg"]["f1-score"]
    recall_macro_train = dict_train["macro avg"]["recall"]
    print(f"F1 Score en entrenamiento (macro avg): {f1_macro_train:.4f}")
    print(f"Recall Score en entrenamiento (macro avg): {recall_macro_train:.4f}")

Distribución total de emociones en el test conflictivo (contando todas):

8: 279
20: 799
1: 754
4: 1181
3: 1145
12: 114
6: 566
22: 580
9: 623
27: 1577
16: 42
25: 563
2: 609
7: 910
0: 1576
15: 897
18: 737
13: 389
5: 487
17: 667
26: 394
10: 692
24: 204
11: 342
14: 179
23: 69
19: 90
21: 69

Entrenando modelo: Logistic Regression

Precisión en validación (Logistic Regression): 0.5346
F1 Score en validación (macro avg): 0.2956
Recall Score en validación (macro avg): 0.2539
Precisión en entrenamiento (Logistic Regression): 0.6003
F1 Score en entrenamiento (macro avg): 0.3932
Recall Score en entrenamiento (macro avg): 0.3299

Entrenando modelo: SVM (linear)
